# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Ranking Signal Analysis.**

I'm picking this lane over the scoring/clustering lanes because it's the one question that has to
be answered *before* any of the others matter: before I build a ranked queue (Lane 2) or score
CTR gaps (Lane 4), I want evidence for which signals actually move with visibility, clicks, and
decline at all — instead of assuming FlyRank's existing intuitions (position matters, freshness
matters, content type matters) hold in this data. It also fits how I want to spend Week 1-2:
I'd rather spend the early weeks doing careful EDA and defending effect sizes than committing to
a model architecture on day one. If the signal work turns up something strong and directional
(e.g. a clean CTR-by-position relationship, or a real freshness effect), that naturally feeds a
Lane 2-style scoring capstone later — so this choice doesn't box me out of ranking/scoring, it
sets it up on firmer ground.


In [1]:
# Nothing to compute yet -- this section is the framing, not the evidence.
# (Real numbers backing this choice are in Section 3.)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Which safe, observable content and search signals (position, freshness,
content type, engagement) are actually associated with visibility, clicks, or decline in this
data -- and how strong are those associations, really?

- **Unit of analysis:** a content page (one row = one pseudonymized content item, trailing 90-day
  window).
- **Decision this improves:** which signals are worth building a scoring model or a ranked
  queue around later in the track. Right now the team could pour Weeks 3-7 into a model built on
  a signal (say, word count) that turns out to barely matter -- this lane exists to catch that
  before it happens.
- **Who acts, and how:** me, as the next decision in *my own* capstone -- and indirectly, an SEO
  content strategist deciding which signal categories (position/CTR, freshness, content type)
  deserve limited review time each week. If I show freshness barely predicts decline but position
  strongly predicts CTR, that changes what a reviewer checks first.
- **The action:** choosing which signal(s) become the backbone of the Lane 2/4-style scoring work
  in later weeks, and which "common wisdom" signals to deprioritize.
- **Cost of a wrong call:** two-sided. If I claim a weak signal is strong, later weeks get built
  on a shaky foundation and the eventual model wastes a reviewer's time chasing noise. If I claim
  a real signal is weak (dismiss it too early), a genuinely useful lever gets left out of the
  final scoring model. Since nothing here is causal or acted on directly by a client yet, the
  immediate cost is wasted internship weeks, not wasted client money -- but the discipline is the
  same discipline a real signal report would need.
- **Why data/ML helps at all:** some of this (e.g. "does CTR fall as position gets worse") is a
  hunch anyone in SEO already holds. The value isn't discovering that it's true -- it's measuring
  *how much*, with a real effect size, on this specific inventory, instead of assuming the
  textbook relationship transfers unchanged. That's a measurement problem, not one that needs a
  complex model: plain grouped comparisons and correlations are the right tool here, and I'd only
  reach for something heavier if a relationship turned out to be genuinely tangled across many
  signals at once.


In [2]:
# Framing only -- no computation needed for this section.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three signals, computed live below on the starter dataset (30,000 rows, `impressions_90d >= 100`
and `avg_position > 0` where noted, per the data dictionary's "0 = no data" gotcha):

1. **Position -> CTR is real and large.** Mean CTR falls from 0.355 at `page_1` down to 0.055 at
   `deep` -- roughly a 6x gap between the best and worst position tier. `corr(avg_position, ctr)`
   on visible pages is -0.239: real, negative, but far from a perfect relationship, which is
   itself useful to know (position explains some but not all of CTR).
2. **Freshness -> decline is *not* the clean story I expected.** The share of pages trending
   "down" is 51.1% for pages updated in the last 30 days, rises to 58.9% and 61.1% for pages
   stale 31-180 days, then drops back down to 47.1% for the *most* stale pages (181+ days). If
   freshness were a simple lever, I'd expect a monotonic climb -- instead the oldest pages look
   fine, which is exactly the kind of "common wisdom doesn't fully hold" finding this lane is
   supposed to surface.
3. **Content type looks thin on its own.** Engagement rate barely differs between the two
   largest content types with enough sessions to compare (`feedly article` 3.02 vs
   `keyword article` 2.71) -- not nothing, but not the kind of gap I'd build a whole feature
   around without checking it against position/freshness first.


In [3]:
import os
import pandas as pd

# Locate the repo root regardless of where this notebook is launched from
# (matches the pattern used in notebooks/01 and notebooks/02).
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df)} rows, {df['client_id'].nunique()} clients\n")

# 1) CTR by position tier -- excludes avg_position == 0 ("no data", per the data dictionary)
visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("Mean CTR by position tier (impressions_90d >= 100):")
print(ctr_by_pos.round(3).to_string())
print(f"\ncorr(avg_position, ctr) on visible pages (n={len(visible)}): "
      f"{visible['avg_position'].corr(visible['ctr']):.3f}\n")

# 2) Freshness tier vs share of pages trending down
down_rate = df.groupby("freshness_tier")["trend_direction"].apply(lambda s: (s == "down").mean())
print("Share of pages trending 'down', by freshness_tier:")
print(down_rate.round(3).to_string())
print()

# 3) Engagement rate by content type (only types with enough sessions to trust the mean)
eng_by_type = (
    df[df["sessions_90d"] >= 30]
    .groupby("content_type")["engagement_rate"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)
print("Mean engagement_rate by content_type (sessions_90d >= 30):")
print(eng_by_type.round(2).to_string())


30000 rows, 32 clients

Mean CTR by position tier (impressions_90d >= 100):
position_tier
page_1      0.355
top_3       0.334
striking    0.256
page_3_5    0.142
deep        0.055

corr(avg_position, ctr) on visible pages (n=22006): -0.239

Share of pages trending 'down', by freshness_tier:
freshness_tier
0-30      0.511
181+      0.471
31-90     0.589
91-180    0.611

Mean engagement_rate by content_type (sessions_90d >= 30):
                 mean  count
content_type                
feedly article   3.02     42
keyword article  2.71   7072


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:** these are *observed, correlational* patterns in one 30,000-row anonymized
snapshot across 32 clients (and later, when I move to the warehouse, in a larger but still
observational panel). I can say a signal is "associated with," "moves together with," or
"directionally linked to" an outcome, and I can attach an effect size (a 6x CTR gap between
position tiers; a ~10-point spread in down-rate across freshness tiers) so the strength of the
claim is visible, not vague.

**What I can't claim:**
- **Not causal.** I did not run an experiment. I cannot say improving a page's position *causes*
  more clicks, or that refreshing a stale page *would* fix a decline -- only that pages in
  better positions currently get more clicks, and staleness currently correlates (weakly and
  non-monotonically) with decline.
- **Not "I proved a Google ranking factor."** I'm looking at outcomes on FlyRank's client
  content, not reverse-engineering search algorithm internals.
- **Not a guarantee for any single page.** These are population-level patterns; any one page can
  (and will) break the trend.
- **The freshness result stays a flag, not a conclusion.** Because it's non-monotonic, I'm
  treating it as "worth investigating further with the warehouse's longer history," not as a
  settled finding I'd build a scoring rule on yet.

Going forward I'll describe results as **observed** (measured directly, e.g. the CTR-by-tier
means), **directional** (a real but noisy relationship, e.g. the freshness pattern), or
**decision-support** (useful for prioritizing review, never a standalone verdict on a page).


In [4]:
# No additional computation needed -- Section 4 is the claims/caveats framing,
# supported by the numbers already computed in Section 3.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.